In [ ]:
from __future__ import annotations
from pathlib import Path
import pandas as pd
import numpy as np
from itertools import combinations
import matplotlib.pyplot as plt
from scipy.signal import periodogram
from scipy.stats import ks_2samp, wasserstein_distance
from scripts.dataImport import fetch_datasets, load_flow, load_packet
import seaborn as sns

Couldn't figure out how to make projects work so here's the outline we discussed today:

Have a nice weekend - Mack
 
**Goal:** Create a notebook that the group can use to decide what features to keep, and how to treat the features we keep. Have this ready for a team discussion by June 30th at the latest.

Tasks:
 - Reading original paper or watching the attached youtube video to identify features that we don't need (this can be found at the link in the original project pdf from the professor)
 - Notebook draft (see below)

What's in the notebook draft?
 - Improve presentation (more Markdown, change layout)
 - Explain metrics (so far we have a measure of mutual information, distribution comparisons (the violin plots), correlation, statistical comparison (KS statistic and Wasserstein distance), structural profiling, and a time series/frequency view (only for packet data)). We can update this list as needed
 - Draft Interpretation of results (to get the ball rolling when we discuss with the whole team)

In [ ]:
fetch_datasets()

In [ ]:

allFlowData = {
    'benign':load_flow("benign"),
    'ddos':load_flow("ddos_http"),
    'dos':load_flow("dos_http"),
    'dns':load_flow("dns_spoofing"),
    'xss':load_flow("xss"),
    'bruteForce':load_flow('brute_force')
}
allPacketData = {
    'benign':load_packet("benign"),
    'ddos':load_packet("ddos_http"),
    'dos':load_packet("dos_http"),
    'dns':load_packet("dns_spoofing"),
    'xss':load_packet("xss"),
    'bruteForce':load_packet('brute_force')
}
print("Done.")


In [ ]:
"""
Step 1 — Structural profiling.

Replaces the original dataStructure() function. Instead of writing one text
file per dataset, this returns an in-memory dict so downstream steps (time
series, violin, stats, redundancy, MI) can pull the column lists they need
without re-scanning the data or parsing text files.
"""
# from __future__ import annotations


def profile_dataset(df: pd.DataFrame, exclude_cols: list[str]) -> dict:
    """
    Profile a single dataset.

    Parameters
    ----------
    df : the raw dataframe for one dataset (e.g. dataDict["benign"])
    exclude_cols : columns that are numeric-dtype but semantically identifiers
        (e.g. flowNumericLabels / packetNumericLabels) — excluded from the
        "continuous numeric feature" treatment (no mean/var/violin/KS on these),
        but still recorded in the profile in case you need them later
        (e.g. Dst Port for mutual information, per the earlier discussion).

    Returns
    -------
    dict with:
        shape: (n_rows, n_cols)
        features: {col_name: {dtype, n_missing, pct_missing, n_inf, is_numeric,
                               is_excluded, mean, var}}
        numeric_cols: numeric dtype, NOT in exclude_cols
        nonzero_var_cols: subset of numeric_cols with var > 0 (computed with
            +/-inf excluded -- see note below)
        zero_var_cols: subset of numeric_cols with var == 0 (candidates to drop)
        excluded_numeric_cols: numeric dtype but in exclude_cols (identifiers)
        non_numeric_cols: everything else (object/datetime/etc.)

    Note on inf: rate-style columns (e.g. 'Flow Bytes/s' = bytes / duration)
    commonly contain +/-inf when duration == 0. mean()/var() over inf-
    contaminated data produces NaN, which previously caused such columns to
    be misclassified as zero-variance and silently dropped from the whole
    pipeline -- even though they likely have real variance once inf rows are
    excluded. mean/var here are computed with inf treated as missing. n_inf
    is reported per feature so you can see which columns are affected and
    decide whether to investigate the underlying zero-duration rows
    separately (that condition can itself be informative, e.g. for
    single-packet probe flows).
    """
    n_rows = len(df)
    features = {}
    numeric_cols, nonzero_var_cols, zero_var_cols = [], [], []
    excluded_numeric_cols, non_numeric_cols = [], []

    for c in df.columns:
        col = df[c]
        n_missing = int(col.isna().sum())
        is_numeric = pd.api.types.is_numeric_dtype(col)
        is_excluded = c in exclude_cols
        n_inf = 0

        entry = {
            "dtype": str(col.dtype),
            "n_missing": n_missing,
            "pct_missing": round(100 * n_missing / n_rows, 4) if n_rows else 0.0,
            "n_inf": 0,
            "is_numeric": is_numeric,
            "is_excluded": is_excluded,
            "mean": None,
            "var": None,
        }

        if is_numeric:
            if is_excluded:
                excluded_numeric_cols.append(c)
            else:
                numeric_cols.append(c)
                finite_col = col.replace([np.inf, -np.inf], np.nan)
                n_inf = int(np.isinf(col).sum())
                entry["n_inf"] = n_inf
                var = float(finite_col.var())
                mean = float(finite_col.mean())
                entry["mean"] = mean
                entry["var"] = var
                if var and var > 0 and not pd.isna(var):
                    nonzero_var_cols.append(c)
                else:
                    zero_var_cols.append(c)
        else:
            non_numeric_cols.append(c)

        features[c] = entry

    return {
        "shape": df.shape,
        "features": features,
        "numeric_cols": numeric_cols,
        "nonzero_var_cols": nonzero_var_cols,
        "zero_var_cols": zero_var_cols,
        "excluded_numeric_cols": excluded_numeric_cols,
        "non_numeric_cols": non_numeric_cols,
    }


def profile_all(dataDict: dict[str, pd.DataFrame], exclude_cols: list[str]) -> dict:
    """Run profile_dataset across every dataset in dataDict. Returns {name: profile_dict}."""
    out = {}
    for name, df in dataDict.items():
        out[name] = profile_dataset(df, exclude_cols)
        missing_feats = [
            c for c, info in out[name]["features"].items() if info["n_missing"] > 0
        ]
        if missing_feats:
            print(f"[{name}] {len(missing_feats)} feature(s) have missing values: {missing_feats}")
        inf_feats = {
            c: info["n_inf"] for c, info in out[name]["features"].items() if info["n_inf"] > 0
        }
        if inf_feats:
            print(f"[{name}] {len(inf_feats)} feature(s) have +/-inf values (mean/var computed "
                  f"excluding inf): {inf_feats}")
        n_zero = len(out[name]["zero_var_cols"])
        if n_zero:
            print(f"[{name}] {n_zero} numeric feature(s) have zero variance: {out[name]['zero_var_cols']}")
    return out


In [ ]:
"""
Packet-only step — Time series + frequency view.

For each non-zero-variance feature: a time-bucketed aggregate plot (mean +
count per time bucket, fast even at 1M+ rows) and an FFT/periodogram to spot
periodicity (e.g. beaconing). Both are cheap because they operate on bucketed
or array data, not raw per-row scatter — see the earlier discussion about why
sns.lineplot(hue="Flow ID") was slow.
"""
# from __future__ import annotations

def time_bucket_plot(
    df: pd.DataFrame,
    feature: str,
    timestamp_col: str = "Timestamp",
    freq: str = "1min",
    ax=None,
):
    """
    Plot mean(feature) and row count per time bucket. Use this instead of
    raw scatter/line on every row — aggregating first is both faster and
    more informative for spotting bursts/drift.
    """
    ts = pd.to_datetime(df[timestamp_col])
    s = df[feature]
    binned = pd.DataFrame({feature: s.values, "_ts": ts.values}).set_index("_ts")
    agg = binned.resample(freq)[feature].agg(["mean", "count"])

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 3.5))
    ax2 = ax.twinx()
    ax.plot(agg.index, agg["count"], color="steelblue", label="row count")
    ax2.plot(agg.index, agg["mean"], color="indianred", alpha=0.7, label=f"mean({feature})")
    ax.set_ylabel("count", color="steelblue")
    ax2.set_ylabel(f"mean {feature}", color="indianred")
    ax.set_title(f"{feature} over time (bucketed: {freq})")
    return ax


def feature_periodogram(df: pd.DataFrame, feature: str, fs: float = 1.0, ax=None):
    """
    Periodogram (power spectral density) for a feature, to spot periodicity
    (e.g. regular beaconing intervals). fs = samples per unit time; if you
    haven't resampled to a uniform time grid, treat the x-axis as cycles
    per sample rather than a calibrated frequency.
    """
    values = df[feature].dropna().values
    if len(values) < 2:
        return None
    values = values - np.mean(values)
    freqs, power = periodogram(values, fs=fs)

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 3))
    ax.semilogy(freqs[1:], power[1:])  # skip DC component at freq=0
    ax.set_xlabel("frequency")
    ax.set_ylabel("power")
    ax.set_title(f"Periodogram: {feature}")
    return ax


def run_time_series_step(
    dataDict: dict[str, pd.DataFrame],
    nonzero_var_cols: dict[str, list[str]],
    timestamp_col: str = "Timestamp",
    freq: str = "1min",
    save_dir=None,
):
    """
    Run time-bucket + periodogram plots for every non-zero-variance feature,
    for every dataset. nonzero_var_cols: {dataset_name: [col, col, ...]}
    (the output of profile_dataset / profile_all from structure.py).

    Returns nothing by default (plots render inline); pass save_dir (a Path)
    to also save PNGs instead of/alongside inline display.
    """
    for name, df in dataDict.items():
        cols = nonzero_var_cols.get(name, [])
        for feature in cols:
            if not pd.api.types.is_numeric_dtype(df[feature]):
                continue
            fig, axes = plt.subplots(2, 1, figsize=(10, 6))
            time_bucket_plot(df, feature, timestamp_col=timestamp_col, freq=freq, ax=axes[0])
            feature_periodogram(df, feature, ax=axes[1])
            fig.suptitle(f"{name} — {feature}")
            fig.tight_layout()
            if save_dir is not None:
                out = save_dir / name
                out.mkdir(parents=True, exist_ok=True)
                fig.savefig(out / f"{feature}_timeseries.png", dpi=100)
                plt.close(fig)
            else:
                plt.show()


In [ ]:
"""
Step 3 — Distributional comparison (violin plots).

One violin plot per feature, showing all datasets side by side (benign +
each attack type as separate groups via `hue`/x-axis category) so you can
compare shape/spread/center at a glance.
"""
# from __future__ import annotations

def build_long_frame(
    dataDict: dict[str, pd.DataFrame],
    feature: str,
    sample_n: int | None = 50_000,
    random_state: int = 0,
) -> pd.DataFrame:
    """
    Stack one feature from every dataset into a single long-format frame:
    columns = [dataset, value]. Optionally sample down large datasets first
    (seaborn's violin KDE doesn't need every row, and 1M+ row datasets like
    your dos packet data will otherwise slow this down for no real accuracy
    gain).
    """
    frames = []
    for name, df in dataDict.items():
        if feature not in df.columns:
            continue
        s = df[feature].dropna()
        if sample_n is not None and len(s) > sample_n:
            s = s.sample(n=sample_n, random_state=random_state)
        frames.append(pd.DataFrame({"dataset": name, "value": s.values}))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["dataset", "value"])


def plot_violin_comparison(
    dataDict: dict[str, pd.DataFrame],
    feature: str,
    sample_n: int | None = 50_000,
    ax=None,
):
    """Violin plot of `feature`, one violin per dataset."""
    long_df = build_long_frame(dataDict, feature, sample_n=sample_n)
    if long_df.empty:
        print(f"No data found for feature '{feature}' in any dataset.")
        return None

    if ax is None:
        fig, ax = plt.subplots(figsize=(max(6, 1.2 * long_df["dataset"].nunique()), 4))
    sns.violinplot(data=long_df, x="dataset", y="value", ax=ax, cut=0)
    ax.set_title(feature)
    ax.tick_params(axis="x", rotation=30)
    return ax


def run_violin_step(
    dataDict: dict[str, pd.DataFrame],
    features: list[str],
    sample_n: int | None = 50_000,
    save_dir=None,
):
    """Run plot_violin_comparison for every feature in `features` (e.g. the
    union or intersection of nonzero_var_cols across datasets)."""
    for feature in features:
        ax = plot_violin_comparison(dataDict, feature, sample_n=sample_n)
        if ax is None:
            continue
        if save_dir is not None:
            save_dir.mkdir(parents=True, exist_ok=True)
            ax.figure.savefig(save_dir / f"{feature}_violin.png", dpi=100, bbox_inches="tight")
            plt.close(ax.figure)
        else:
            plt.show()


In [ ]:
"""
Step 4 — Statistical comparison (all pairwise combinations of datasets, per
feature).

Outputs a long-format DataFrame: one row per (feature, dataset_a, dataset_b)
with the KS statistic and Wasserstein distance. Long format instead of a wide
cross-tab because with 6 datasets that's C(6,2) = 15 pairs per feature —
much easier to filter/sort/pivot from long format than wrangle a giant
multi-index wide matrix.

KS p-value is NOT included as a primary ranking signal: at 50k-1M+ rows
almost every comparison will be "significant" regardless of practical size
of the difference (see earlier discussion). Wasserstein distance is the
recommended primary ranking metric since it's in the same units as the
feature itself and scales with how different the distributions actually are.
"""
# from __future__ import annotations
def compare_pair(s_a: pd.Series, s_b: pd.Series) -> dict:
    """
    Note: +/-inf values (e.g. from 'Flow Bytes/s' when Flow Duration == 0)
    are dropped here the same as NaN. Left in, wasserstein_distance silently
    returns inf, which would dominate any ranking built on top of this
    (see top_separating_features). KS test also switches calculation modes
    silently when inf is present. n_a/n_b report counts AFTER this drop, so
    a big gap vs. the original dataset size is your signal that a feature
    has a lot of inf/NaN worth investigating on its own.
    """
    a = pd.Series(s_a).replace([np.inf, -np.inf], np.nan).dropna().values
    b = pd.Series(s_b).replace([np.inf, -np.inf], np.nan).dropna().values
    if len(a) == 0 or len(b) == 0:
        return {"ks_stat": None, "ks_pvalue": None, "wasserstein": None,
                "n_a": len(a), "n_b": len(b)}
    ks_stat, ks_p = ks_2samp(a, b)
    wd = wasserstein_distance(a, b)
    return {"ks_stat": ks_stat, "ks_pvalue": ks_p, "wasserstein": wd,
            "n_a": len(a), "n_b": len(b)}


def run_statistical_comparison(
    dataDict: dict[str, pd.DataFrame],
    features: list[str],
) -> pd.DataFrame:
    """
    All-pairwise statistical comparison across datasets for each feature.

    Returns a long DataFrame:
        feature, dataset_a, dataset_b, ks_stat, ks_pvalue, wasserstein, n_a, n_b
    """
    rows = []
    dataset_names = list(dataDict.keys())

    for feature in features:
        for name_a, name_b in combinations(dataset_names, 2):
            df_a, df_b = dataDict[name_a], dataDict[name_b]
            if feature not in df_a.columns or feature not in df_b.columns:
                continue
            result = compare_pair(df_a[feature], df_b[feature])
            rows.append({
                "feature": feature,
                "dataset_a": name_a,
                "dataset_b": name_b,
                **result,
            })

    return pd.DataFrame(rows)


def top_separating_features(stats_df: pd.DataFrame, n: int = 20, metric: str = "wasserstein") -> pd.DataFrame:
    """
    Convenience helper: average a metric (default Wasserstein distance) across
    all pairs per feature, sorted descending — a quick "which features differ
    most across datasets overall" ranking.

    Note: Wasserstein distance isn't normalized across features with different
    scales/units, so use this for ranking within a feature's own comparisons,
    or normalize first if comparing across features of very different scale
    (e.g. Flow Duration in microseconds vs a 0-1 ratio feature).
    """
    return (
        stats_df.groupby("feature")[metric]
        .mean()
        .sort_values(ascending=False)
        .head(n)
        .reset_index()
        .rename(columns={metric: f"mean_{metric}"})
    )


def top_separating_features_normalized(
    dataDict: dict[str, pd.DataFrame],
    stats_df: pd.DataFrame,
    n: int = 20,
) -> pd.DataFrame:
    """
    Same idea as top_separating_features, but divides each feature's mean
    Wasserstein distance by that feature's overall standard deviation (pooled
    across all datasets) before ranking. This puts features on a comparable
    scale so e.g. 'Flow Duration' (microseconds, large numbers) doesn't
    automatically dominate over a 0-1 ratio feature just because of units.
    Use this version when comparing separation power ACROSS features; use the
    raw version when comparing across dataset-pairs WITHIN one feature.
    """
    rows = []
    for feature, group in stats_df.groupby("feature"):
        all_vals = pd.concat([df[feature] for df in dataDict.values() if feature in df.columns])
        all_vals = all_vals.replace([float("inf"), float("-inf")], pd.NA).dropna()
        std = all_vals.std()
        if not std or std == 0 or pd.isna(std):
            continue
        mean_wd = group["wasserstein"].mean()
        rows.append({"feature": feature, "mean_wasserstein": mean_wd,
                      "pooled_std": std, "normalized_wasserstein": mean_wd / std})

    out = pd.DataFrame(rows).sort_values("normalized_wasserstein", ascending=False).head(n)
    return out.reset_index(drop=True)


In [ ]:
"""
Step 5 — Redundancy check.

Correlation matrix per dataset to catch near-duplicate features (common in
CICFlowMeter-style exports, e.g. Total Fwd Packet vs Subflow Fwd Packets).
Run per-dataset (redundancy is a property of the feature set/data generation
process, not of benign vs attack, so there's no pairwise-across-datasets
version needed here — unlike Step 4).
"""
# from __future__ import annotations

def compute_correlation_matrix(df: pd.DataFrame, cols: list[str], method: str = "pearson") -> pd.DataFrame:
    """
    Correlation matrix restricted to `cols` (pass nonzero_var_cols, excluding
    identifiers).

    Note: +/-inf is replaced with NaN before correlating. pandas .corr() does
    NOT raise on inf -- it silently produces a numerically wrong-but-plausible
    result (verified: a column containing inf can come back with correlation
    1.0 against another column, which is misleading, not just imprecise).
    inf typically comes from rate columns like 'Flow Bytes/s' when Flow
    Duration == 0.
    """
    valid_cols = [c for c in cols if c in df.columns]
    clean = df[valid_cols].replace([np.inf, -np.inf], np.nan)
    return clean.corr(method=method)


def find_redundant_pairs(corr: pd.DataFrame, threshold: float = 0.9) -> pd.DataFrame:
    """
    Long-format list of feature pairs with |correlation| >= threshold,
    sorted descending. This is the practical output you act on — e.g. drop
    one of each pair before modeling, or note them for dimensionality
    reduction.
    """
    rows = []
    cols = corr.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            c_val = corr.iloc[i, j]
            if pd.notna(c_val) and abs(c_val) >= threshold:
                rows.append({"feature_a": cols[i], "feature_b": cols[j], "correlation": c_val})

    if not rows:
        return pd.DataFrame(columns=["feature_a", "feature_b", "correlation"])

    return pd.DataFrame(rows).sort_values("correlation", key=abs, ascending=False).reset_index(drop=True)


def plot_correlation_heatmap(corr: pd.DataFrame, title: str = "", ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(min(20, 0.4 * len(corr)), min(20, 0.4 * len(corr))))
    sns.heatmap(corr, cmap="coolwarm", center=0, square=True, ax=ax,
                cbar_kws={"shrink": 0.6}, xticklabels=True, yticklabels=True)
    ax.set_title(title)
    return ax


def run_redundancy_step(
    dataDict: dict[str, pd.DataFrame],
    nonzero_var_cols: dict[str, list[str]],
    threshold: float = 0.9,
    save_dir=None,
) -> dict[str, pd.DataFrame]:
    """
    Run correlation + redundant-pair detection for every dataset.
    Returns {dataset_name: redundant_pairs_df}.
    """
    results = {}
    for name, df in dataDict.items():
        cols = nonzero_var_cols.get(name, [])
        corr = compute_correlation_matrix(df, cols)
        redundant = find_redundant_pairs(corr, threshold=threshold)
        results[name] = redundant

        ax = plot_correlation_heatmap(corr, title=f"{name} — correlation")
        if save_dir is not None:
            save_dir.mkdir(parents=True, exist_ok=True)
            ax.figure.savefig(save_dir / f"{name}_correlation.png", dpi=100, bbox_inches="tight")
            plt.close(ax.figure)
        else:
            plt.show()

        if not redundant.empty:
            print(f"[{name}] {len(redundant)} redundant pair(s) at |r| >= {threshold}")

    return results


In [ ]:
"""
Step 6 — Feature ranking via mutual information.

Builds a combined (benign + all attack) frame with a label column, then
computes mutual information between each feature and the label. This is the
main "which features actually separate benign from attack" payoff step,
independent of any model's distributional assumptions.

Two label modes:
  - binary: benign vs attack (everything non-benign collapsed to 1 label)
  - multiclass: benign vs ddos vs dos vs dns vs xss vs bruteForce, etc.
Both are useful for different questions (binary = pure IDS, multiclass =
attack-type classification), so this computes both by default.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif


def build_labeled_frame(
    dataDict: dict[str, pd.DataFrame],
    features: list[str],
    benign_key: str = "benign",
    sample_n: int | None = 100_000,
    random_state: int = 0,
) -> pd.DataFrame:
    """
    Stack all datasets into one frame with a 'label_binary' (benign/attack)
    and 'label_multiclass' (dataset name) column, restricted to `features`.
    Samples each dataset down to sample_n rows if larger, to keep the MI
    computation tractable at your scale (dos alone is 1.6M flow rows).
    """
    frames = []
    for name, df in dataDict.items():
        valid_cols = [c for c in features if c in df.columns]
        sub = df[valid_cols].copy()
        if sample_n is not None and len(sub) > sample_n:
            sub = sub.sample(n=sample_n, random_state=random_state)
        sub["label_multiclass"] = name
        sub["label_binary"] = "benign" if name == benign_key else "attack"
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)


def compute_mutual_information(
    labeled_df: pd.DataFrame,
    features: list[str],
    label_col: str = "label_binary",
    random_state: int = 0,
) -> pd.DataFrame:
    """
    Mutual information between each feature and label_col.

    Handles +/-inf (common in CICFlowMeter-derived rate columns like
    'Flow Bytes/s' or 'Flow Packets/s' when Flow Duration == 0 -- divide by
    zero produces inf, not NaN, and sklearn rejects inf outright) by treating
    it the same as NaN: rows with inf/NaN in ANY of the selected features are
    dropped for this MI computation only (the original labeled_df/dataDict
    are untouched).

    Note: inf here is often itself informative (e.g. a flow with 0 duration
    but nonzero bytes -- a single-packet probe). Dropping it for MI doesn't
    erase that signal from your dataset, just from this particular ranking;
    consider separately flagging "Flow Duration == 0" as its own feature if
    it differs meaningfully between benign and attack.

    Returns a DataFrame sorted descending by MI score.
    """
    valid_cols = [c for c in features if c in labeled_df.columns]
    subset = labeled_df[valid_cols + [label_col]].copy()

    # Replace +/-inf with NaN so dropna() catches both in one pass
    numeric_cols = [c for c in valid_cols if pd.api.types.is_numeric_dtype(subset[c])]
    subset[numeric_cols] = subset[numeric_cols].replace([np.inf, -np.inf], np.nan)

    n_before = len(subset)
    clean = subset.dropna()
    n_after = len(clean)
    n_dropped = n_before - n_after
    if n_dropped > 0:
        pct = round(100 * n_dropped / n_before, 2) if n_before else 0.0
        print(f"[mutual_info:{label_col}] dropped {n_dropped}/{n_before} rows ({pct}%) "
              f"due to inf/NaN in one or more features")

    if clean.empty:
        return pd.DataFrame(columns=["feature", "mutual_info"])

    X = clean[valid_cols]
    y = clean[label_col]

    mi = mutual_info_classif(X, y, random_state=random_state)
    return (
        pd.DataFrame({"feature": valid_cols, "mutual_info": mi})
        .sort_values("mutual_info", ascending=False)
        .reset_index(drop=True)
    )


def run_mutual_information_step(
    dataDict: dict[str, pd.DataFrame],
    features: list[str],
    benign_key: str = "benign",
    sample_n: int | None = 100_000,
) -> dict[str, pd.DataFrame]:
    """
    Convenience wrapper: builds the labeled frame once, computes both binary
    and multiclass MI rankings. Returns {"binary": df, "multiclass": df}.
    """
    labeled = build_labeled_frame(dataDict, features, benign_key=benign_key, sample_n=sample_n)
    return {
        "binary": compute_mutual_information(labeled, features, label_col="label_binary"),
        "multiclass": compute_mutual_information(labeled, features, label_col="label_multiclass"),
    }


In [ ]:
"""
Pipeline drivers.

run_packet_pipeline: Step 1 (structure) -> time series/frequency -> Step 3
    (violin) -> Step 4 (stats) -> Step 5 (redundancy) -> Step 6 (mutual info)
run_flow_pipeline:    Step 1 (structure) -> Step 3 -> Step 4 -> Step 5 -> Step 6
    (no time series/frequency step — flow data is already aggregated per-flow,
    so per-row time series isn't meaningful the same way; see earlier discussion)

Results are kept in a SEPARATE resultsDict, NOT merged into dataDict, so the
original dataframes stay untouched and reusable.
"""
from __future__ import annotations


def _common_nonzero_var_features(profiles: dict) -> list[str]:
    """
    Union vs intersection matters here: a feature with zero variance in ONE
    dataset (e.g. a flag that's always 0 in benign traffic but varies under
    attack) is exactly the kind of feature you want to catch, not discard.
    So we take the UNION of nonzero_var_cols across datasets, not the
    intersection — a feature is included if it has signal in at least one
    dataset.
    """
    all_cols = set()
    for p in profiles.values():
        all_cols.update(p["nonzero_var_cols"])
    return sorted(all_cols)


def run_packet_pipeline(
    dataDict: dict[str, pd.DataFrame],
    exclude_cols: list[str],
    benign_key: str = "benign",
    timestamp_col: str = "Timestamp",
    save_root: Path | None = None,
) -> dict:
    results = {}

    print("=== Step 1: structural profiling ===")
    profiles = profile_all(dataDict, exclude_cols)
    results["profiles"] = profiles
    nonzero_var_by_dataset = {name: p["nonzero_var_cols"] for name, p in profiles.items()}
    features = _common_nonzero_var_features(profiles)
    results["features_used"] = features
    print(f"{len(features)} feature(s) with nonzero variance in at least one dataset")

    ts_dir = (save_root / "timeseries") if save_root else None
    print("=== Time series + frequency view (packet-only) ===")
    run_time_series_step(dataDict, nonzero_var_by_dataset, timestamp_col=timestamp_col, save_dir=ts_dir)

    violin_dir = (save_root / "violin") if save_root else None
    print("=== Step 3: violin comparison ===")
    run_violin_step(dataDict, features, save_dir=violin_dir)

    print("=== Step 4: statistical comparison (all pairwise) ===")
    stats_df = run_statistical_comparison(dataDict, features)
    results["stats"] = stats_df
    results["top_separating_features"] = top_separating_features_normalized(dataDict, stats_df)

    redundancy_dir = (save_root / "redundancy") if save_root else None
    print("=== Step 5: redundancy check ===")
    results["redundancy"] = run_redundancy_step(dataDict, nonzero_var_by_dataset, save_dir=redundancy_dir)

    print("=== Step 6: mutual information ranking ===")
    results["mutual_info"] = run_mutual_information_step(dataDict, features, benign_key=benign_key)

    return results


def run_flow_pipeline(
    dataDict: dict[str, pd.DataFrame],
    exclude_cols: list[str],
    benign_key: str = "benign",
    save_root: Path | None = None,
) -> dict:
    results = {}

    print("=== Step 1: structural profiling ===")
    profiles = profile_all(dataDict, exclude_cols)
    results["profiles"] = profiles
    nonzero_var_by_dataset = {name: p["nonzero_var_cols"] for name, p in profiles.items()}
    features = _common_nonzero_var_features(profiles)
    results["features_used"] = features
    print(f"{len(features)} feature(s) with nonzero variance in at least one dataset")

    violin_dir = (save_root / "violin") if save_root else None
    print("=== Step 3: violin comparison ===")
    run_violin_step(dataDict, features, save_dir=violin_dir)

    print("=== Step 4: statistical comparison (all pairwise) ===")
    stats_df = run_statistical_comparison(dataDict, features)
    results["stats"] = stats_df
    results["top_separating_features"] = top_separating_features_normalized(dataDict, stats_df)

    redundancy_dir = (save_root / "redundancy") if save_root else None
    print("=== Step 5: redundancy check ===")
    results["redundancy"] = run_redundancy_step(dataDict, nonzero_var_by_dataset, save_dir=redundancy_dir)

    print("=== Step 6: mutual information ranking ===")
    results["mutual_info"] = run_mutual_information_step(dataDict, features, benign_key=benign_key)

    return results

In [ ]:
flowNumericLables = ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp']

In [ ]:
packetNumericLables = ['stream', 'src_mac', 'dst_mac', 'src_ip', 'dst_ip', 'src_port', 'dst_port']

In [ ]:
results = run_flow_pipeline(dataDict=allFlowData,exclude_cols=flowNumericLables)

In [ ]:
results.keys()

In [ ]:
results

In [ ]:
results['top_separating_features']

TCP/IP Stack (Addressing): Flow ID, Src IP, Src Port, Dst IP, Dst Port, Protocol, 
FlowID: Unique ID for the packet exchange
Timestamp: 

## General Dataset Metrics
The key idea is to compare the benign data set to each attack data set on a feature by feature basis. This will allow us to:
 - Identify useful features for 
 - Drop non-useful features
 - 
Comparing Features in 3 Ways:
 - Compare the mean, variance, and distribution of each feature